# 22 — Despliegue: el modelo de 5 variables en la web
Punto **4** del informe. Un modelo solo sirve si alguien lo puede usar. Aquí se toma la versión de **5 variables** del
notebook 21 y se exporta para que una página web calcule **cuántos días tardaría la fila de un proceso**, en el navegador,
**sin servidor y sin instalar nada**. La página para la gente (`despliegue/sitio/`) la arma el notebook 23; la versión
de clase (`web/prediccion.html`) deja escribir los números a mano.

Cómo se logra:
1. Se elige el mejor modelo de 5 variables **entre los que se pueden exportar** a un archivo de datos: los de árboles
   (Bagging, Random Forest, Gradient Boosting). Un árbol es solo una lista de preguntas «¿variable ≤ umbral?»; se copian
   esas preguntas y las hojas a un JSON y la página las recorre en JavaScript.
2. Se calcula un **rango** honesto para cada estimación: con validación cruzada se mide cuánto se equivocó el modelo en
   procesos que no vio, y se guarda entre qué errores cayó la mitad central (del 25 % al 75 %). Como un proceso con
   miles de causas es más predecible que uno con 15, el rango se mide por separado según el tamaño.
3. Se guarda el modelo entrenado (`models/modelo_5var.joblib`) y su versión exportada (`web/modelo/modelo_5var.json`),
   con una copia idéntica en `despliegue/sitio/modelo/`.
4. **Control:** una reimplementación que lee **solo el JSON** debe dar exactamente lo mismo que scikit-learn.

In [1]:
%run -i modulos/comun.ipynb
%run -i modulos/modelado.ipynb
import math
import warnings
import joblib
from sklearn.model_selection import cross_val_predict

warnings.filterwarnings("ignore", category=FutureWarning)
datos = cargar_conjunto()
part = pd.read_csv(MODELO_DIR / "particion.csv")
assert np.allclose(part[OBJETIVO].to_numpy(), datos[OBJETIVO].to_numpy())
tr = part.index[part["particion"] == "entrenamiento"]
te = part.index[part["particion"] == "prueba"]
y_tr, y_te = datos.loc[tr, OBJETIVO], datos.loc[te, OBJETIVO]
hp = leer_hp()
TOP5 = json.loads((MODELO_DIR / "top5.json").read_text(encoding="utf-8"))["variables"]
versiones = pd.read_csv(MODELO_DIR / "comparacion_versiones.csv")
v5 = versiones[versiones["version"] == "5 más importantes"].set_index("modelo")
EXPORTABLES = ["Bagging", "Random Forest", "Gradient Boosting"]
ELEGIDO = v5.loc[EXPORTABLES, "mae_log_cv"].idxmin()
display(v5[["mae_log_cv", "mae_log_prueba", "dentro_doble_prueba"]].sort_values("mae_log_cv").round(3))
print("variables:", TOP5)
print("modelo desplegado:", ELEGIDO, "(menor error de validación cruzada entre los exportables)")

,mae_log_cv,mae_log_prueba,dentro_doble_prueba
modelo,,,
Stacking,0.770,0.716,0.698
SVM,0.773,0.718,0.710
Gradient Boosting,0.794,0.725,0.679
Regresión lineal (Ridge),0.800,0.695,0.667
Random Forest,0.807,0.753,0.617
AdaBoost,0.814,0.767,0.648
Bagging,0.820,0.756,0.630


variables: ['log_stock_inicial', 'presion_stock', 'log_ingresos', 'ingresos_por_juzgado', 'stock_por_juzgado']
modelo desplegado: Gradient Boosting (menor error de validación cruzada entre los exportables)


## Entrenar y guardar
Se entrena con el 80 % de entrenamiento (el mismo con que se evaluó), para que las métricas de la prueba sigan siendo
las del modelo publicado.

In [2]:
final = modelo(ELEGIDO, TOP5, hp).fit(datos.loc[tr, TOP5], y_tr)
pred_te = final.predict(datos.loc[te, TOP5])
metricas_final = medir(y_te, pred_te)
print({k: round(v, 3) for k, v in metricas_final.items()})
assert abs(metricas_final["mae_log"] - v5.loc[ELEGIDO, "mae_log_prueba"]) < 1e-9, "no es el mismo modelo del notebook 21"
CARPETA_MODELOS = RAIZ / "models"
CARPETA_MODELOS.mkdir(exist_ok=True)
joblib.dump(final, CARPETA_MODELOS / "modelo_5var.joblib")

{'mae_log': np.float64(0.725), 'factor_mediano': 1.584, 'dentro_doble': 0.679, 'r2_log': np.float64(0.384), 'error_dias_mediano': 80.824}


['C:\\Users\\barry\\maxi\\u\\analisis\\congestion-judicial-bolivia\\models\\modelo_5var.joblib']

## El rango: cuánto se equivoca en procesos que no vio
Predicciones *fuera de pliegue* en el entrenamiento (cada proceso estimado por un modelo que no lo vio). El error es
`real − estimado` en log: un error de +0,4 significa que el tiempo real fue $e^{0,4} \approx 1,5$ veces el estimado.
Se guardan los cuantiles 25 % y 75 % por tamaño del proceso: el **rango probable** de la página. Después se comprueba en
la **prueba**, que no participó: si el rango está bien hecho, alrededor de la mitad de los procesos de prueba debería
caer dentro.

In [3]:
TAMANOS = [MIN_CAUSAS, 31, 101, 501]          # límites inferiores: 10-30, 31-100, 101-500, más de 500
ETIQ_TAMANOS = ["10 a 30 causas", "31 a 100 causas", "101 a 500 causas", "más de 500 causas"]


def clase_tamano(atendidas):
    return np.searchsorted(TAMANOS, np.asarray(atendidas, float), side="right") - 1


oof = cross_val_predict(modelo(ELEGIDO, TOP5, hp), datos.loc[tr, TOP5], y_tr, cv=CV5, n_jobs=-1)
residuo = y_tr.to_numpy() - oof
clase_tr = clase_tamano(datos.loc[tr, "atendidas"])
bandas = []
for k, etiqueta in enumerate(ETIQ_TAMANOS):
    r = residuo[clase_tr == k]
    bandas.append({"desde": TAMANOS[k], "etiqueta": etiqueta, "procesos": int(len(r)),
                   "q25": float(np.quantile(r, 0.25)), "q75": float(np.quantile(r, 0.75)),
                   "q10": float(np.quantile(r, 0.10)), "q90": float(np.quantile(r, 0.90))})
bandas_df = pd.DataFrame(bandas)
clase_te = clase_tamano(datos.loc[te, "atendidas"])
res_te = y_te.to_numpy() - pred_te
q25 = bandas_df["q25"].to_numpy()[clase_te]
q75 = bandas_df["q75"].to_numpy()[clase_te]
dentro = (res_te >= q25) & (res_te <= q75)
bandas_df["cobertura_prueba"] = [dentro[clase_te == k].mean() for k in range(len(TAMANOS))]
bandas_df["procesos_prueba"] = [int((clase_te == k).sum()) for k in range(len(TAMANOS))]
bandas_df["factor_ancho"] = np.exp(bandas_df["q75"] - bandas_df["q25"])
bandas_df.to_csv(MODELO_DIR / "bandas.csv", index=False)
display(bandas_df.round(3))
cobertura = float(dentro.mean())
print(f"en la prueba, {dentro.sum()} de {len(te)} procesos ({cobertura:.0%}) cayeron dentro de su rango probable")

,desde,etiqueta,procesos,q25,q75,q10,q90,cobertura_prueba,procesos_prueba,factor_ancho
0,10,10 a 30 causas,163,-0.847,0.855,-1.667,1.573,0.581,43,5.484
1,31,31 a 100 causas,197,-0.522,0.585,-1.395,1.529,0.500,48,3.027
2,101,101 a 500 causas,177,-0.358,0.445,-1.177,0.878,0.514,37,2.232
3,501,más de 500 causas,109,-0.411,0.367,-0.678,0.720,0.706,34,2.177


en la prueba, 92 de 162 procesos (57%) cayeron dentro de su rango probable


## Exportar a JSON
El JSON lleva: qué datos pedir (las columnas originales que necesitan las 5 variables), cómo calcular las variables de
ingeniería, el preprocesamiento (mediana para faltantes, categorías del one-hot), los árboles y las bandas del rango.

In [4]:
entradas = []
for c in TOP5:
    for dep in DEPENDENCIAS.get(c, [c]):
        if dep not in entradas:
            entradas.append(dep)
desc_entradas = []
for c in entradas:
    if c in CATEGORICAS:
        desc_entradas.append({"columna": c, "nombre": NOMBRES[c], "tipo": "categorica",
                              "opciones": sorted(datos.loc[tr, c].dropna().unique().tolist()),
                              "moda": datos.loc[tr, c].mode().iloc[0]})
    else:
        v = datos.loc[tr, c]
        desc_entradas.append({"columna": c, "nombre": NOMBRES[c], "tipo": "numerica", "min": float(v.min()),
                              "mediana": float(v.median()), "p90": float(v.quantile(0.9)), "max": float(v.max())})

pre = final.named_steps["pre"]
num_cols = list(pre.transformers_[0][2])
num_pipe = pre.named_transformers_["num"]
pre_json = {"numericas": num_cols, "medianas": num_pipe.named_steps["imputar"].statistics_.tolist(),
            "escala": None, "categoricas": [], "categorias": []}
assert "escalar" not in num_pipe.named_steps, "los modelos de árboles no usan escala"
if "cat" in pre.named_transformers_:
    pre_json["categoricas"] = list(pre.transformers_[1][2])
    pre_json["categorias"] = [c.tolist() for c in pre.named_transformers_["cat"].named_steps["onehot"].categories_]


def arbol_json(arbol):
    t = arbol.tree_
    return {"izq": t.children_left.tolist(), "der": t.children_right.tolist(), "var": t.feature.tolist(),
            "umbral": t.threshold.tolist(), "valor": t.value[:, 0, 0].tolist()}


est = final.named_steps["modelo"]
Xt_te = pre.transform(datos.loc[te, TOP5])
if ELEGIDO == "Random Forest":
    modelo_json = {"tipo": "promedio_arboles", "arboles": [arbol_json(a) for a in est.estimators_], "subconjuntos": None}
elif ELEGIDO == "Bagging":
    modelo_json = {"tipo": "promedio_arboles", "arboles": [arbol_json(a) for a in est.estimators_],
                   "subconjuntos": [s.tolist() for s in est.estimators_features_]}
else:
    suma = sum(a[0].predict(Xt_te.astype(np.float32)) for a in est.estimators_)
    inicial = est.predict(Xt_te) - est.learning_rate * suma
    assert np.ptp(inicial) < 1e-9, "el valor inicial del boosting debería ser constante"
    modelo_json = {"tipo": "boosting", "inicial": float(inicial[0]), "tasa_aprendizaje": est.learning_rate,
                   "arboles": [arbol_json(a[0]) for a in est.estimators_]}

muestra = datos.loc[te].sample(20, random_state=SEMILLA)
autoverificacion = [{"entrada": {c: (r[c] if c in CATEGORICAS else float(r[c])) for c in entradas},
                     "log_tiempo": float(final.predict(muestra.loc[[i], TOP5])[0]),
                     "dias_real": float(r[DIAS]), "proceso": r["tipo_proceso"], "territorio": r["territorio"],
                     "materia": r["materia_homologada"]}
                    for i, r in muestra.iterrows()]
v28 = versiones[versiones["version"] == "con ingeniería e historia (28)"].set_index("modelo")
export = {
    "descripcion": "Tiempo estimado de la fila de un proceso no penal: log(1 + días) = pendientes al cierre / resueltas x 365",
    "objetivo": "log1p_dias", "tope_dias": TOPE_DIAS, "min_causas": MIN_CAUSAS,
    "modelo": ELEGIDO, "hiperparametros": hp[ELEGIDO], "variables": TOP5,
    "nombres": {c: NOMBRES[c] for c in TOP5 + entradas}, "formulas": {c: FORMULAS[c] for c in TOP5 if c in FORMULAS},
    "entradas": desc_entradas, "preprocesamiento": pre_json,
    "bandas": bandas_df[["desde", "etiqueta", "q25", "q75", "q10", "q90", "cobertura_prueba"]].to_dict("records"),
    "metricas_prueba": metricas_final,
    "comparacion": {"mae_log_prueba_28_variables": float(v28.loc[ELEGIDO, "mae_log_prueba"]),
                    "mae_log_cv_5_variables": float(v5.loc[ELEGIDO, "mae_log_cv"]),
                    "cobertura_rango_prueba": cobertura},
    "n_entrenamiento": int(len(tr)), "n_prueba": int(len(te)), "autoverificacion": autoverificacion,
    "arbol_compara_float32": True, **modelo_json,
}
(RAIZ / "web" / "modelo").mkdir(exist_ok=True)
ruta_json = RAIZ / "web" / "modelo" / "modelo_5var.json"
ruta_json.write_text(json.dumps(export, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
ruta_despliegue = RAIZ / "despliegue" / "sitio" / "modelo" / "modelo_5var.json"
ruta_despliegue.parent.mkdir(parents=True, exist_ok=True)
ruta_despliegue.write_bytes(ruta_json.read_bytes())
assert ruta_despliegue.read_bytes() == ruta_json.read_bytes()
print(f"{ruta_json.name}: {ruta_json.stat().st_size / 1e6:.2f} MB, {len(modelo_json['arboles'])} árboles; entradas: {entradas}")

modelo_5var.json: 0.11 MB, 100 árboles; entradas: ['pendientes_inicio_efectivo', 'ingresos_totales', 'num_juzgados_pagina']


## Control: predecir leyendo solo el JSON
Esta función no usa scikit-learn: hace a mano lo mismo que hará la página (calcular las variables nuevas, imputar,
one-hot, recorrer los árboles). Los árboles de scikit-learn comparan los datos convertidos a `float32`; la reimplementación
también, o un dato justo en el umbral podría ir por la rama equivocada.

In [5]:
J = json.loads(ruta_json.read_text(encoding="utf-8"))


def variables_desde_entradas(e):
    s, i, j = e.get("pendientes_inicio_efectivo"), e.get("ingresos_totales"), e.get("num_juzgados_pagina")
    calc = {
        "log_stock_inicial": lambda: math.log1p(s),
        "log_ingresos": lambda: math.log1p(i),
        "presion_stock": lambda: s / (i + 1),
        "ingresos_por_juzgado": lambda: i / j,
        "stock_por_juzgado": lambda: s / j,
        "prop_acefalias": lambda: e["personal_items_acefalias"] / e["personal_items_total"],
        "prop_excusa": lambda: e["recibidas_excusa_recusacion"] / (i + 1),
        "sin_stock_inicial": lambda: 1.0 if s == 0 else 0.0,
    }
    return {c: (calc[c]() if c in calc else e[c]) for c in J["variables"]}


def recorrer(arbol, x):
    n = 0
    while arbol["izq"][n] != -1:
        n = arbol["izq"][n] if float(np.float32(x[arbol["var"][n]])) <= arbol["umbral"][n] else arbol["der"][n]
    return arbol["valor"][n]


def predecir_json(e):
    """log(1 + días), como scikit-learn."""
    v = variables_desde_entradas(e)
    p = J["preprocesamiento"]
    x = [v[c] if v[c] is not None and not (isinstance(v[c], float) and math.isnan(v[c])) else med
         for c, med in zip(p["numericas"], p["medianas"])]
    for c, cats in zip(p["categoricas"], p["categorias"]):
        x += [1.0 if v[c] == k else 0.0 for k in cats]
    if J["tipo"] == "promedio_arboles":
        subs = J["subconjuntos"] or [None] * len(J["arboles"])
        return sum(recorrer(a, x if s is None else [x[k] for k in s]) for a, s in zip(J["arboles"], subs)) / len(J["arboles"])
    return J["inicial"] + J["tasa_aprendizaje"] * sum(recorrer(a, x) for a in J["arboles"])


p_json = np.array([predecir_json({c: datos.loc[k, c] for c in entradas}) for k in te])
dif = np.abs(p_json - pred_te).max()
print(f"diferencia máxima JSON vs scikit-learn en {len(te)} procesos de prueba: {dif:.2e}")
assert dif < 1e-9
assert all(abs(predecir_json(a["entrada"]) - a["log_tiempo"]) < 1e-9 for a in J["autoverificacion"])

diferencia máxima JSON vs scikit-learn en 162 procesos de prueba: 4.44e-15


## Tablas y cifras para el informe LaTeX
Se escriben desde los CSV de los notebooks 19 a 22 a `informe/tablas/`, para que el informe no copie cifras a mano.

In [6]:
TABLAS = RAIZ / "informe" / "tablas"
TABLAS.mkdir(parents=True, exist_ok=True)


def tex(s):
    s = str(s)
    for a, b in [("\\", r"\textbackslash{}"), ("_", r"\_"), ("%", r"\%"), ("&", r"\&"), ("#", r"\#"), ("$", r"\$")]:
        s = s.replace(a, b)
    return s


def num(x, d=3):
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and float(x).is_integer() and d == 0):
        return f"{int(x):,}".replace(",", ".")
    if d == 0:
        return f"{round(x):,}".replace(",", ".")
    return f"{x:.{d}f}".replace(".", ",")


def pct(x, d=0):
    return num(100 * x, d)


def tabla(nombre, encabezado, filas, columnas):
    cuerpo = "\n".join(" & ".join(f) + r" \\" for f in filas)
    (TABLAS / f"{nombre}.tex").write_text(
        f"\\begin{{tabular}}{{{columnas}}}\n\\toprule\n{' & '.join(encabezado)} \\\\\n\\midrule\n{cuerpo}\n\\bottomrule\n\\end{{tabular}}\n",
        encoding="utf-8")


variables = pd.read_csv(MODELO_DIR / "variables.csv").fillna("")
corr = pd.read_csv(MODELO_DIR / "eda_correlaciones.csv").set_index("variable")
mod = variables[variables["grupo"] != "excluida (fuga)"]
tabla("variables", ["Variable", "Descripción", "Grupo", "Conversión", r"$\rho$ Spearman"],
      [[r"\texttt{" + tex(r.variable) + "}", tex(r.nombre if not r.formula else r.formula), tex(r.grupo),
        tex(r.conversion.replace(" + escala (SVM)", "").replace("imputación mediana", "mediana")),
        num(corr.loc[r.variable, "spearman"], 2) if r.variable in corr.index else "---"]
       for r in mod.itertuples()], "lp{5.2cm}llr")
fuga = variables[variables["grupo"] == "excluida (fuga)"]
tabla("fuga", ["Columna excluida", "Motivo"],
      [[r"\texttt{" + tex(r.variable) + "}", tex(r.motivo_exclusion)] for r in fuga.itertuples()], "lp{9cm}")

mcv = pd.read_csv(MODELO_DIR / "metricas_cv.csv")
pv = mcv.pivot_table(index="modelo", columns="configuracion", values="mae_log").reindex(MODELOS)
aj = mcv[mcv["configuracion"] == "ajustado"].set_index("modelo").reindex(MODELOS)
tabla("metricas_cv", ["Modelo", "Familia", "MAE por defecto", "MAE ajustado", "Desvío", "Dentro del doble", r"$R^2$"],
      [[tex(m), tex(FAMILIAS[m][0]), num(pv.loc[m, "por defecto"]), num(pv.loc[m, "ajustado"]),
        num(aj.loc[m, "mae_log_desvio"]), pct(aj.loc[m, "dentro_doble"]) + r"\,\%", num(aj.loc[m, "r2_log"])]
       for m in MODELOS], "llrrrrr")


def valor_hp(v):
    """Valor legible: None = sin límite, enteros sin decimales, decimales con 3 cifras significativas."""
    if v is None:
        return "sin límite"
    if isinstance(v, bool) or isinstance(v, str):
        return tex(v)
    if float(v).is_integer():
        return num(int(v), 0)
    return f"{v:.3g}".replace(".", ",")


tabla("hiperparametros", ["Modelo", "Hiperparámetro", "Valor elegido"],
      [[tex(m), r"\texttt{" + tex(k) + "}", valor_hp(v)] for m, p in hp.items() for k, v in p.items()], "lll")

mt = pd.read_csv(MODELO_DIR / "metricas_test.csv").set_index("modelo").reindex(MODELOS)
tabla("metricas_test", ["Modelo", "MAE (log)", "Dentro del doble", "Factor mediano", r"$R^2$", "Error mediano (días)"],
      [[tex(m), num(r.mae_log), pct(r.dentro_doble) + r"\,\%", r"$\times$" + num(r.factor_mediano, 2), num(r.r2_log),
        num(r.error_dias_mediano, 0)] for m, r in mt.iterrows()], "lrrrrr")

imp = pd.read_csv(MODELO_DIR / "importancia.csv")
tabla("importancia", ["\\#", "Variable", "Grupo", "Permutación", "Desvío", "Impureza RF"],
      [[str(r.rango), tex(r.nombre), tex(r.grupo), num(r.importancia, 4), num(r.desvio_entre_pliegues, 4),
        num(r.importancia_impureza_rf, 3)] for r in imp.head(12).itertuples()], "rllrrr")

ver = pd.read_csv(MODELO_DIR / "comparacion_versiones.csv")
orden_v = ["originales (19)", "con ingeniería e historia (28)", "5 más importantes"]
pcv = ver.pivot_table(index="modelo", columns="version", values="mae_log_cv")
pte = ver.pivot_table(index="modelo", columns="version", values="mae_log_prueba")
mods = [m for m in MODELOS if m in pcv.index]
tabla("versiones", ["Modelo", "19 (CV)", "19 (prueba)", "28 (CV)", "28 (prueba)", "5 (CV)", "5 (prueba)"],
      [[tex(m)] + [num(x) for v in orden_v for x in (pcv.loc[m, v], pte.loc[m, v])] for m in mods], "lrrrrrr")

rob = pd.read_csv(MODELO_DIR / "robustez_grupos.csv")
tabla("robustez", ["Versión", "Modelo", "MAE CV normal", "MAE por grupos", "Diferencia"],
      [[tex(r.version), tex(r.modelo), num(r.mae_log_cv_normal), num(r.mae_log_por_grupos), num(r.diferencia)]
       for r in rob.itertuples()], "llrrr")

tam = pd.read_csv(MODELO_DIR / "prueba_por_tamano.csv").set_index("tamano").reindex(["10-30", "31-100", "101-500", "más de 500"])
tabla("tamano", ["Causas atendidas", "Procesos de prueba", "MAE (log)", "Dentro del doble"],
      [[tex(t), num(int(r.procesos), 0), num(r.mae_log), pct(r.dentro_doble) + r"\,\%"] for t, r in tam.iterrows()], "lrrr")

tabla("bandas", ["Tamaño del proceso", "Procesos (entren.)", r"Rango: $\times$ desde", r"$\times$ hasta",
                 "Dentro en la prueba"],
      [[tex(r.etiqueta), num(int(r.procesos), 0), num(math.exp(r.q25), 2), num(math.exp(r.q75), 2),
        f"{int(round(r.cobertura_prueba * r.procesos_prueba))} de {int(r.procesos_prueba)}"]
       for r in bandas_df.itertuples()], "lrrrr")

formas = pd.read_csv(MODELO_DIR / "historia_formas.csv", index_col=0)
tabla("historia_formas", ["Forma de anticipar el tiempo del año", "Error mediano", "Error medio", "Sesgo", "Peor año"],
      [[tex(f), num(r.error_abs_mediano_pct, 1) + r"\,\%", num(r.error_abs_medio_pct, 1) + r"\,\%",
        num(r.sesgo_pct, 1) + r"\,\%", num(r.peor_pct, 1) + r"\,\%"] for f, r in formas.iterrows()], "lrrrr")

vac = pd.read_csv(MODELO_DIR / "historia_vaciado.csv")
vn = vac[vac["ambito"] == "nacional"]
tabla("historia_vaciado", ["Año", "Tiempo estimado (días)", "Lo que de verdad tardó (días)", "Diferencia"],
      [[str(int(r.gestion)), num(r.tiempo_estimado, 0), num(r.tiempo_real_vaciado, 0),
        ("+" if r.error_pct > 0 else "") + num(r.error_pct, 0) + r"\,\%"] for r in vn.itertuples()], "rrrr")
print(sorted(p.name for p in TABLAS.glob("*.tex")))

['bandas.tex', 'cifras.tex', 'fuga.tex', 'hiperparametros.tex', 'historia_formas.tex', 'historia_vaciado.tex', 'importancia.tex', 'metricas_cv.tex', 'metricas_test.tex', 'robustez.tex', 'tamano.tex', 'variables.tex', 'versiones.tex']


In [7]:
for viejo in ["matriz_confusion.csv"]:
    (MODELO_DIR / viejo).unlink(missing_ok=True)       # del modelo de clasificación anterior
mejor_cv = aj.loc[ENTRENABLES, "mae_log"].idxmin()
tiempos = pd.read_csv(MODELO_DIR / "historia_tiempos.csv").set_index("gestion")
hm = pd.read_csv(MODELO_DIR / "historia_materias.csv")
y_all = datos[DIAS]
procesos = cargar_procesos()
macros = {
    "nProcesos": num(len(procesos), 0), "nGrandes": num(int((procesos["atendidas"] >= MIN_CAUSAS).sum()), 0),
    "nChicos": num(int((procesos["atendidas"] < MIN_CAUSAS).sum()), 0),
    "nSinResueltas": num(int(((procesos["atendidas"] >= MIN_CAUSAS) & (procesos["resueltas"] == 0)).sum()), 0),
    "nFilas": num(len(datos), 0), "nEntrenamiento": num(len(tr), 0), "nPrueba": num(len(te), 0),
    "minCausas": str(MIN_CAUSAS), "medianaCausas": num(int(procesos["atendidas"].median()), 0),
    "medianaDias": num(y_all.median(), 0), "pveinticincoDias": num(y_all.quantile(0.25), 0),
    "psetentaycincoDias": num(y_all.quantile(0.75), 0), "nTopados": num(int((procesos.loc[(procesos["atendidas"] >= MIN_CAUSAS) & (procesos["resueltas"] > 0), "duracion_estimada_dias"].astype(float) > TOPE_DIAS).sum()), 0),
    "nTiposProceso": num(datos["tipo_proceso"].nunique(), 0),
    "nOriginales": str(len(ORIGINALES)), "nIngenieria": str(len(INGENIERIA)), "nHistoricas": str(len(HISTORICAS)),
    "nCompleta": str(len(COMPLETA)), "nFuga": str(len(FUGA)),
    "mejorModelo": tex(mejor_cv), "mejorMaeCv": num(aj.loc[mejor_cv, "mae_log"]),
    "mejorMaePrueba": num(mt.loc[mejor_cv, "mae_log"]), "mejorDobPrueba": pct(mt.loc[mejor_cv, "dentro_doble"]),
    "mejorFactorPrueba": num(mt.loc[mejor_cv, "factor_mediano"], 2), "mejorRdosPrueba": num(mt.loc[mejor_cv, "r2_log"], 2),
    "mejorDiasPrueba": num(mt.loc[mejor_cv, "error_dias_mediano"], 0),
    "modeloDesplegado": tex(ELEGIDO), "maeDesplegado": num(metricas_final["mae_log"]),
    "dobDesplegado": pct(metricas_final["dentro_doble"]), "factorDesplegado": num(metricas_final["factor_mediano"], 2),
    "rdosDesplegado": num(metricas_final["r2_log"], 2), "diasDesplegado": num(metricas_final["error_dias_mediano"], 0),
    "maeDesplegadoVeintiocho": num(export["comparacion"]["mae_log_prueba_28_variables"]),
    "coberturaRango": pct(cobertura), "nArboles": str(len(modelo_json["arboles"])),
    "pesoJson": num(ruta_json.stat().st_size / 1e6, 1),
    "difJson": "0" if dif == 0 else f"{dif:.0e}".replace("e-", r"\times10^{-") + "}",
    "nEntradas": str(len(entradas)),
    # historia
    "histErrVaciado": num(vac["error_pct"].abs().median(), 1), "histErrVaciadoMedio": num(vac["error_pct"].abs().mean(), 1),
    "histSesgoVaciado": num(vac["error_pct"].median(), 1), "histNVaciado": num(len(vac), 0),
    "histErrAnterior": num(formas.loc["el del año anterior", "error_abs_mediano_pct"], 1),
    "histPeorAnterior": num(formas.loc["el del año anterior", "peor_pct"], 0),
    "histErrIngenua": num(formas.loc["fórmula ingenua", "error_abs_mediano_pct"], 1),
    "histSesgoIngenua": num(formas.loc["fórmula ingenua", "sesgo_pct"], 1),
    "histErrRitmo": num(formas.loc["stock ÷ ritmo anterior", "error_abs_mediano_pct"], 1),
    "histTiempoDosmilsiete": num(tiempos.loc[2007, "nacional"], 0), "histTiempoDosmildoce": num(tiempos.loc[2012, "nacional"], 0),
    "histTiempoDosmilquince": num(tiempos.loc[2015, "nacional"], 0), "histTiempoDosmilveinte": num(tiempos.loc[2020, "nacional"], 0),
    "histTiempoDosmilveintitres": num(tiempos.loc[2023, "nacional"], 0),
    "histCapitalDosmilveintitres": num(tiempos.loc[2023, "capital"], 0),
    "histProvinciaDosmilveintitres": num(tiempos.loc[2023, "provincia"], 0),
}
for k, c in enumerate(TOP5, 1):
    macros[f"topVar{'ABCDE'[k - 1]}"] = tex(NOMBRES[c])
CLAVE_MODELO = {"Línea base (promedio)": "Base", "Fórmula ingenua": "Ingenua", "Regresión lineal (Ridge)": "Ridge",
                "SVM": "SVM", "Bagging": "Bagging", "Random Forest": "RandomForest", "AdaBoost": "AdaBoost",
                "Gradient Boosting": "GradientBoosting", "Stacking": "Stacking"}
CLAVE_VERSION = {"originales (19)": "Diecinueve", "con ingeniería e historia (28)": "Veintiocho", "5 más importantes": "Cinco"}
for m, k in CLAVE_MODELO.items():
    macros[f"maeDef{k}"] = num(pv.loc[m, "por defecto"])
    macros[f"maeCv{k}"] = num(aj.loc[m, "mae_log"])
    macros[f"maePrueba{k}"] = num(mt.loc[m, "mae_log"])
    macros[f"dobPrueba{k}"] = pct(mt.loc[m, "dentro_doble"])
    macros[f"rdosPrueba{k}"] = num(mt.loc[m, "r2_log"], 2)
    if m in pcv.index:
        for v, kv in CLAVE_VERSION.items():
            macros[f"maeV{kv}Cv{k}"] = num(pcv.loc[m, v])
            macros[f"maeV{kv}Prueba{k}"] = num(pte.loc[m, v])
for r in rob.itertuples():
    macros[f"maeGrupos{CLAVE_VERSION[r.version]}{CLAVE_MODELO[r.modelo]}"] = num(r.mae_log_por_grupos)
    macros[f"difGrupos{CLAVE_VERSION[r.version]}{CLAVE_MODELO[r.modelo]}"] = num(r.diferencia)
for t, k in zip(["10-30", "31-100", "101-500", "más de 500"], ["A", "B", "C", "D"]):
    macros[f"dobTamano{k}"] = pct(tam.loc[t, "dentro_doble"])
imp_i = imp.set_index("variable")
macros |= {
    "impHistorica": num(imp_i.loc["tendencia_resueltas_materia", "importancia"], 4),
    "rangoHistorica": str(int(imp_i.loc["tendencia_resueltas_materia", "rango"])),
    "svmC": num(hp["SVM"]["C"], 0), "svmGamma": num(hp["SVM"]["gamma"], 3), "svmEpsilon": num(hp["SVM"]["epsilon"], 1),
    "maeSvmMin": num(pd.read_csv(MODELO_DIR / "grilla_svm.csv", index_col=0).values.min()),
    "maeSvmMax": num(pd.read_csv(MODELO_DIR / "grilla_svm.csv", index_col=0).values.max()),
    "rhoPresion": num(corr.loc["presion_stock", "spearman"], 2), "rhoIngresos": num(corr.loc["ingresos_totales", "spearman"], 2),
    "rhoStock": num(corr.loc["pendientes_inicio_efectivo", "spearman"], 2),
    "rhoHistorica": num(corr.loc["tendencia_resueltas_materia", "spearman"], 2),
    "rangoChicoDesde": num(math.exp(bandas_df.loc[0, "q25"]), 2), "rangoChicoHasta": num(math.exp(bandas_df.loc[0, "q75"]), 2),
    "rangoGrandeDesde": num(math.exp(bandas_df.loc[3, "q25"]), 2), "rangoGrandeHasta": num(math.exp(bandas_df.loc[3, "q75"]), 2),
}
lineas = [f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in macros.items()]
(TABLAS / "cifras.tex").write_text("% Generado por notebooks/22_despliegue.ipynb\n" + "\n".join(lineas) + "\n", encoding="utf-8")
print("\n".join(lineas))

\newcommand{\nProcesos}{1.240}
\newcommand{\nGrandes}{818}
\newcommand{\nChicos}{422}
\newcommand{\nSinResueltas}{10}
\newcommand{\nFilas}{808}
\newcommand{\nEntrenamiento}{646}
\newcommand{\nPrueba}{162}
\newcommand{\minCausas}{10}
\newcommand{\medianaCausas}{28}
\newcommand{\medianaDias}{221}
\newcommand{\pveinticincoDias}{100}
\newcommand{\psetentaycincoDias}{503}
\newcommand{\nTopados}{13}
\newcommand{\nTiposProceso}{78}
\newcommand{\nOriginales}{19}
\newcommand{\nIngenieria}{8}
\newcommand{\nHistoricas}{1}
\newcommand{\nCompleta}{28}
\newcommand{\nFuga}{19}
\newcommand{\mejorModelo}{Stacking}
\newcommand{\mejorMaeCv}{0,738}
\newcommand{\mejorMaePrueba}{0,694}
\newcommand{\mejorDobPrueba}{67}
\newcommand{\mejorFactorPrueba}{1,52}
\newcommand{\mejorRdosPrueba}{0,41}
\newcommand{\mejorDiasPrueba}{87}
\newcommand{\modeloDesplegado}{Gradient Boosting}
\newcommand{\maeDesplegado}{0,725}
\newcommand{\dobDesplegado}{68}
\newcommand{\factorDesplegado}{1,58}
\newcommand{\rdosDesplegado}{0,3

## Lectura
- **Modelo publicado: Gradient Boosting con 5 variables** (100 árboles en secuencia, pérdida Huber). Entre los exportables fue
  el de menor error de validación cruzada con 5 variables (0,794; Random Forest 0,807, Bagging 0,820). Stacking y SVM
  quedaron un poco mejor (0,770 y 0,773), pero no se pueden pasar a un JSON de árboles; la diferencia es menor que el ruido.
- **Para usarlo basta con 3 datos:** stock heredado (la fila que espera), ingresos del año y juzgados. Las 5 variables
  (log del stock, presión del stock, log de los ingresos, ingresos por juzgado y stock por juzgado) se calculan con las fórmulas.
- **En la prueba:** error 0,725 en log, **68 %** de los procesos dentro del doble, factor mediano ×1,58, R² 0,38 y error
  mediano de **81 días**. Con las 28 variables el mismo modelo daba 0,710: simplificar a 5 cuesta muy poco.
- **El rango probable es honesto:** se mide con los errores de validación cruzada, por tamaño. En un proceso de 10 a 30
  causas va de ×0,43 a ×2,35 de la estimación; en uno de más de 500, de ×0,66 a ×1,44. En la prueba, que no participó,
  **92 de 162 procesos (57 %)** cayeron dentro de su rango, cerca del 50 % buscado (un poco más en los grandes: 71 %).
- **El JSON es fiel:** la reimplementación sin scikit-learn coincide en los 162 procesos de prueba con una diferencia
  máxima de 4·10⁻¹⁵ (el redondeo de la última cifra de un número de máquina). El archivo pesa 0,1 MB.
- **Las tablas y cifras del informe** quedan en `informe/tablas/` (13 tablas y `cifras.tex`): el `.tex` las incluye con
  `\input`, así que si se vuelve a correr el pipeline, el informe se actualiza solo al compilar.